# II. Clean and Validate Dataset

This notebook starts from the raw dataset acquired in notebook 1 and performs the full cleaning procedure directly in notebook cells.
It then validates the cleaned output and saves `../dataset/marketing_campaign_dataset_cleaned.csv`.

## Step 1: Imports, Paths, and Prerequisite Check

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Keep raw and cleaned paths explicit for lineage purposes.
RAW_DATASET_PATH = Path("../dataset/marketing_campaign_dataset.csv")
CLEANED_DATASET_PATH = Path("../dataset/marketing_campaign_dataset_cleaned.csv")

## Fail-fast guard: stop early if notebook 1 output is missing.
if not RAW_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at: {RAW_DATASET_PATH.resolve()}\n"
        "Run 01_initial_exploration.ipynb first to acquire raw data."
    )

## Display options for better visibility of wide tables during exploration.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"Using raw dataset: {RAW_DATASET_PATH.resolve()}")

Using raw dataset: D:\VSCode\py_marketing_eda\dataset\marketing_campaign_dataset.csv


## Step 2: Load Raw Data and Validate Required Schema

In [2]:
## Keep an immutable raw reference so cleaning can branch from a known baseline.
df_raw = pd.read_csv(RAW_DATASET_PATH)

## Contract-style schema definition.
REQUIRED_COLUMNS = [
    "Campaign_ID",
    "Company",
    "Campaign_Type",
    "Target_Audience",
    "Duration",
    "Channel_Used",
    "Conversion_Rate",
    "Acquisition_Cost",
    "ROI",
    "Location",
    "Language",
    "Clicks",
    "Impressions",
    "Engagement_Score",
    "Customer_Segment",
    "Date",
]

## Error-handling.
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in raw dataset: {missing_columns}")

print(f"Raw shape: {df_raw.shape}")
print("Required schema check passed.")

Raw shape: (200000, 16)
Required schema check passed.


## Step 3: Apply Cleaning Transformations

Transformations implemented in-notebook:
- Clean `Acquisition_Cost` by removing `$` and `,` and converting to float
- Clean `Duration` by removing ` days` and converting to integer
- Parse `Date` as datetime
- Replace infinite values with `NaN`
- Remove duplicate rows

In [3]:
## Work on a copy to preserve `df_raw` for audits/repro comparisons.
df = df_raw.copy()

## Currency normalization pipeline: cast -> strip symbols -> parse numeric.
## `errors="coerce"` converts malformed tokens to NaN instead of failing hard.
acquisition_clean = (
    df["Acquisition_Cost"].astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["Acquisition_Cost"] = pd.to_numeric(acquisition_clean, errors="coerce").astype(float)

## Duration normalization removes unit suffix and keeps nullable integer semantics.
duration_clean = df["Duration"].astype(str).str.replace(" days", "", regex=False).str.strip()
df["Duration"] = pd.to_numeric(duration_clean, errors="coerce").astype("Int64")

## Datetime coercion preserves parse failures as NaT for transparent QA.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

## Replace +/-inf so summary stats and model inputs stay finite.
df = df.replace([np.inf, -np.inf], np.nan)

## Capture dedup impact metrics before resetting a clean sequential index.
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)

print(f"Rows before deduplication: {rows_before}")
print(f"Rows after deduplication:  {rows_after}")
print(f"Duplicates removed:         {rows_before - rows_after}")

Rows before deduplication: 200000
Rows after deduplication:  200000
Duplicates removed:         0


## Step 4: Save Cleaned Dataset

In [4]:
## Ensure output directory exists; avoids failure on first run/new clone.
CLEANED_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

## Persist without index to prevent synthetic row-id leakage into downstream analysis.
df.to_csv(CLEANED_DATASET_PATH, index=False)

print(f"Cleaned CSV saved to: {CLEANED_DATASET_PATH.resolve()}")
print(f"Cleaned DataFrame shape: {df.shape}")

Cleaned CSV saved to: D:\VSCode\py_marketing_eda\dataset\marketing_campaign_dataset_cleaned.csv
Cleaned DataFrame shape: (200000, 16)


## Step 5: Validation and Post-Clean Profiling

In [5]:
## `info(show_counts=True)` combines dtype audit with non-null coverage in one view.
df.info(show_counts=True)

## Sort numeric describe output by `std` to prioritize high-variance features.
numeric_profile = df.describe(include=[np.number]).T.sort_values(by="std", ascending=False)
display(numeric_profile.head(15))

## Missingness is shown as count + percent to compare absolute and relative impact.
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct}).sort_values(
    by=["missing_count", "missing_pct"], ascending=False
)
print("Missing values summary per column:")
display(missing_summary)

## Explicit key-column dtype check validates the intended cleaning outcomes.
key_columns = ["Date", "Duration", "Acquisition_Cost", "ROI", "Conversion_Rate"]
print("\nKey cleaned dtypes:")
print(df[key_columns].dtypes)

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Campaign_ID       200000 non-null  int64         
 1   Company           200000 non-null  str           
 2   Campaign_Type     200000 non-null  str           
 3   Target_Audience   200000 non-null  str           
 4   Duration          200000 non-null  Int64         
 5   Channel_Used      200000 non-null  str           
 6   Conversion_Rate   200000 non-null  float64       
 7   Acquisition_Cost  200000 non-null  float64       
 8   ROI               200000 non-null  float64       
 9   Location          200000 non-null  str           
 10  Language          200000 non-null  str           
 11  Clicks            200000 non-null  int64         
 12  Impressions       200000 non-null  int64         
 13  Engagement_Score  200000 non-null  int64         
 14  Customer_Segmen

,count,mean,std,min,25%,50%,75%,max
Campaign_ID,200000.0,100000.5,57735.171256,1.0,50000.75,100000.5,150000.25,200000.0
Acquisition_Cost,200000.0,12504.39304,4337.664545,5000.0,8739.75,12496.5,16264.0,20000.0
Impressions,200000.0,5507.30152,2596.864286,1000.0,3266.0,5517.5,7753.0,10000.0
Clicks,200000.0,549.77203,260.019056,100.0,325.0,550.0,775.0,1000.0
Duration,200000.0,37.503975,16.74672,15.0,30.0,30.0,45.0,60.0
Engagement_Score,200000.0,5.49471,2.872581,1.0,3.0,5.0,8.0,10.0
ROI,200000.0,5.002438,1.734488,2.0,3.5,5.01,6.51,8.0
Conversion_Rate,200000.0,0.08007,0.040602,0.01,0.05,0.08,0.12,0.15


Missing values summary per column:


,missing_count,missing_pct
Campaign_ID,0,0.0
Company,0,0.0
Campaign_Type,0,0.0
Target_Audience,0,0.0
Duration,0,0.0
Channel_Used,0,0.0
Conversion_Rate,0,0.0
Acquisition_Cost,0,0.0
ROI,0,0.0
Location,0,0.0



Key cleaned dtypes:
Date                datetime64[us]
Duration                     Int64
Acquisition_Cost           float64
ROI                        float64
Conversion_Rate            float64
dtype: object


## Summary

In [6]:
## Lightweight text audit for logs/CI where rich notebook display is unavailable.
# Temporary audit cell for concise notebook-2 insights

## Recompute missingness locally to keep this cell self-contained.
missing_summary_local = (
    pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(4),
    })
    .sort_values(["missing_count", "missing_pct"], ascending=False)
)

## Same ranking criterion as above to keep variance diagnostics consistent.
numeric_profile_local = df.describe(include=[np.number]).T.sort_values("std", ascending=False)

print("ROW_COUNTS")
print(f"rows_before={rows_before}, rows_after={rows_after}, duplicates_removed={rows_before - rows_after}")
print("\nMISSING_TOP")
print(missing_summary_local.head(5).to_string())
print("\nNUMERIC_STD_TOP")
print(numeric_profile_local[["mean", "std", "min", "max"]].head(5).to_string())
print("\nKEY_DTYPES")
print(df[["Date", "Duration", "Acquisition_Cost", "ROI", "Conversion_Rate"]].dtypes.to_string())

ROW_COUNTS
rows_before=200000, rows_after=200000, duplicates_removed=0

MISSING_TOP
                 missing_count  missing_pct
Campaign_ID                  0          0.0
Company                      0          0.0
Campaign_Type                0          0.0
Target_Audience              0          0.0
Duration                     0          0.0

NUMERIC_STD_TOP
                         mean           std     min       max
Campaign_ID          100000.5  57735.171256     1.0  200000.0
Acquisition_Cost  12504.39304   4337.664545  5000.0   20000.0
Impressions        5507.30152   2596.864286  1000.0   10000.0
Clicks              549.77203    260.019056   100.0    1000.0
Duration            37.503975      16.74672    15.0      60.0

KEY_DTYPES
Date                datetime64[us]
Duration                     Int64
Acquisition_Cost           float64
ROI                        float64
Conversion_Rate            float64


Proceed to `03_bivariate_analysis.ipynb` for relationship analysis using the cleaned dataset.